In [1]:
from direct_social_belief_sim import *

In [2]:
import numpy as np

from direct_social_belief_sim import run_exchange, ExchangeConfig

belief_weights = np.full((5, 5), 0.4)
np.fill_diagonal(belief_weights, 0.0)

agent_adjacency = np.array(
    [
        [0, 1, 1, 0],
        [1, 0, 1, 1],
        [1, 1, 0, 1],
        [0, 1, 1, 0],
    ],
    dtype=float,
)

graph, history = run_exchange(
    belief_weights,
    agent_adjacency,
    config=ExchangeConfig(
        beta_internal=1.0,
        beta_social=1.5,
        focal_beliefs=[0, 2, 4],
        n_steps=1000,
        seed=42,
    ),
    show_progress=True,
    return_history=True,
)

print(np.asarray(graph.vs["beliefs"]))
print(history[0]["topic"], history[0]["is_focal"])

Opinion exchange:   0%|          | 0/1000 [00:00<?, ?it/s]

[[-0.66666667 -1.         -1.         -1.         -1.        ]
 [-1.          0.         -0.66666667 -0.66666667 -1.        ]
 [-0.66666667  0.33333333 -1.          0.33333333 -1.        ]
 [-1.         -1.         -1.         -0.33333333 -1.        ]]
1 False


In [3]:
import time
import numpy as np
import igraph as ig
from scipy import sparse

import direct_social_belief_sim as sim_py
import direct_social_belief_sim_cy_sparse as sim_cy


def bench(label, func, config_cls, adjacency, n_steps=10_000, focal_beliefs=None):
    config = config_cls(
        beta_internal=1.0,
        beta_social=1.5,
        focal_beliefs=focal_beliefs,
        n_steps=n_steps,
        seed=42,
    )

    t0 = time.perf_counter()
    graph, _ = func(
        belief_weights,
        adjacency,
        config=config,
        show_progress=False,
    )
    dt = time.perf_counter() - t0

    beliefs = np.asarray(graph.vs["beliefs"])
    print(f"{label:22s} {dt:.4f}s  mean={beliefs.mean():.3f}  shape={beliefs.shape}")
    return graph, dt

In [4]:
n_agents = 1000
n_beliefs = 20
m = 3

g = ig.Graph.Barabasi(n=n_agents, m=m, directed=False)

agent_adjacency = np.array(g.get_adjacency().data, dtype=np.float64)
agent_adjacency = np.maximum(agent_adjacency, agent_adjacency.T)

agent_csr = sparse.csr_matrix(agent_adjacency)

belief_weights = np.full((n_beliefs, n_beliefs), 0.4, dtype=np.float64)
np.fill_diagonal(belief_weights, 0.0)

focal_beliefs = list(range(5))

print(g.vcount(), g.ecount(), agent_csr.nnz)

1000 2994 5988


In [5]:
results = {}

results["python"] = bench(
    "python",
    sim_py.run_exchange,
    sim_py.ExchangeConfig,
    agent_adjacency,
    n_steps=100_000,
    focal_beliefs=focal_beliefs,
)

results["cython_sparse"] = bench(
    "cython sparse",
    sim_cy.run_exchange_fast_sparse,
    sim_cy.ExchangeConfig,
    agent_csr,
    n_steps=100_000,
    focal_beliefs=focal_beliefs,
)

python                 9.6452s  mean=-0.018  shape=(1000, 20)
cython sparse          0.5892s  mean=0.029  shape=(1000, 20)
